# 2주차 · 과거 기록에서 예측까지

예측하기 전에 이용할 수 있는 기록을 고르고, 훈련에서 배운 변환으로 새로운 기록을 처리합니다. 이어서 점수와 임계값, 평가 기준을 살펴봅니다.

이 노트북의 숫자와 표는 설명을 위해 새로 만든 합성 자료입니다. 실제 센서의 성능이나 사물의 관계를 뜻하지 않습니다. 앞의 2주차 교안에서 배운 훈련·검증·테스트, 평균 대체, 표준화와 혼동행렬을 활용합니다.

## 1. 무엇을 언제 알 수 있을까요?

매일 **측정 전에** 개체의 당일 이상 여부를 예측한다고 약속합시다. 시각을 정수 $t$로 나타냅니다. 한 예측 행은 `(개체, t)`입니다.

| 정보 | 예측 시점에 사용? | 이유 |
|---|---|---|
| 개체 종류 | 예 | 설치 시 미리 기록됨 |
| 같은 개체의 시각 $u<t$에 도착한 측정 | 예 | 이미 관찰됨 |
| 시각 $t$의 측정, 이후의 측정 | 아니요 | 측정 전에 예측함 |
| 당일 이상 정답 | 아니요 | 예측 후에만 확인됨 |

정답은 `+1` 이상, `-1` 정상입니다. 각 정답은 그날 끝에 알려집니다. 예측 특성에 넣지 않습니다. 개체·시각·행 ID는 대응 관계를 확인하기 위한 식별 정보입니다. 모델의 수치 입력으로 자동 사용하지 않습니다.

**이 예제의 목표**는 훈련에 없던 새 개체를 이후 시점에 예측하는 것입니다. 따라서 예측 행을 만들기 **전에** 훈련 대상 A/B와 검증 대상 C/D를 정하고, 훈련 목표 시각은 4·5, 검증은 6·7로 나눕니다. 훈련의 정답은 시각 6 이전에 확인되었다고 가정합니다. 별도의 새 개체 E는 최종 테스트용 자리만 정해 두며, 자료나 정답은 여기에 없습니다.

새 개체 C/D의 과거 측정은 이 예측 서비스가 시각 6 전에 이미 받은 **같은 개체의 라벨 없는 문맥**으로 가정합니다. 이것은 A/B의 훈련 예측 행이나 `fit`에 넣지 않습니다. 시각 6 측정은 그날 예측 **후** 도착하여 시각 7 예측에는 허용됩니다. 만약 모든 시각의 예측을 시각 6 이전에 한꺼번에 제출하는 계약이라면 시각 6 측정도 사용할 수 없고 그 계약에 맞게 다시 만들어야 합니다.

목적에 따라 분할은 달라집니다. 이미 알고 있는 개체의 **미래**를 예측한다면 같은 개체가 훈련과 검증 양쪽에 있을 수 있지만 시간 순서를 지켜야 합니다. 처음 보는 개체를 같은 시기의 독립 자료로 평가하는 목적이라면 원본 개체를 겹치지 않게 나누되 반드시 미래 시점을 요구하는 것은 아닙니다. 이 예제처럼 **새 개체의 미래**가 목적일 때만 두 조건을 모두 요구합니다. 어떤 분할도 그 자체로 모든 누수를 막지는 못합니다.

## 2. 같은 개체의 과거로 특성을 만듭니다

아래 원본 측정표에는 예측 뒤에야 알게 되는 값도 보관되어 있습니다. 각 예측 행에 실제로 들어갈 값을 고를 때는 반드시 `같은 개체`와 `측정 시각 < 예측 시각` 조건을 적용합니다.

- `recent`: 허용된 과거를 시간순으로 정렬해 빈 값을 **forward-fill**한 뒤 마지막 값을 고릅니다. 이전 관측이 전혀 없으면 결측입니다.
- `mean3`, `std3`: 빈 값이 아닌 **최근 원본 관측 세 개**를 사용합니다. 빈 칸을 forward-fill한 복사값을 세 번의 관측으로 세지 않습니다. 실제 관측이 세 개 미만이면 두 값 모두 결측입니다.
- 이전 관측 세 값이 $x_1,x_2,x_3$이면 $m=(x_1+x_2+x_3)/3$, 모집단 표준편차 $s=\sqrt{[(x_1-m)^2+(x_2-m)^2+(x_3-m)^2]/3}$입니다. 이 표준편차는 `ddof=0`입니다.

예를 들어 A의 시각 4 예측에서는 앞선 원본 값이 2, 4, 6이므로 `recent=6`, $m=4$, $s=\sqrt{8/3}$입니다. A의 시각 5 예측에서는 시각 4의 값 4까지 도착했으므로 세 관측이 4, 6, 4입니다. 평균은 $14/3$, 표준편차는 $\sqrt{8/9}$입니다. 예측 시각의 측정은 포함하지 않습니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# 원본 측정은 일부러 섞여 있습니다. 시각별 측정은 예측 후 도착합니다.
records = pd.DataFrame([
    ("A", 1, 2.0), ("A", 2, 4.0), ("A", 3, 6.0), ("A", 4, 4.0),
    ("B", 1, np.nan), ("B", 2, np.nan), ("B", 3, np.nan), ("B", 4, 2.0),
    ("C", 1, 3.0), ("C", 2, 5.0), ("C", 3, 8.0), ("C", 4, np.nan), ("C", 6, 12.0),
    ("D", 1, np.nan), ("D", 2, 1.0), ("D", 3, np.nan), ("D", 4, np.nan), ("D", 6, 7.0),
], columns=["unit", "time", "reading"]).sample(frac=1, random_state=17).reset_index(drop=True)

train_targets = pd.DataFrame([
    ("a4", "A", 4, "red", -1), ("a5", "A", 5, "red", 1),
    ("b4", "B", 4, "blue", -1), ("b5", "B", 5, "blue", 1),
], columns=["id", "unit", "time", "kind", "label"])
val_targets = pd.DataFrame([
    ("c6", "C", 6, "green", 1), ("c7", "C", 7, "green", -1),
    ("d6", "D", 6, "red", -1), ("d7", "D", 7, "red", 1),
], columns=train_targets.columns)

assert set(train_targets.unit).isdisjoint(val_targets.unit)
assert train_targets.time.max() < val_targets.time.min()
assert not records.duplicated(["unit", "time"]).any()

In [ ]:
def make_features(raw, targets):
    # 한 개체에 같은 시각의 기록이 둘이면 실제 도착 순서를 정해야 합니다.
    if raw.duplicated(["unit", "time"]).any():
        raise ValueError("개체와 시각이 중복됩니다.")
    rows = []
    for row in targets.itertuples(index=False):
        past = raw.loc[(raw.unit == row.unit) & (raw.time < row.time)]
        past = past.sort_values("time")  # groupby만으로 시간 정렬이 되지는 않습니다.
        filled = past.reading.ffill()
        recent = filled.iloc[-1] if len(filled) else np.nan
        last3 = past.reading.dropna().tail(3).to_numpy(dtype=float)
        mean3 = last3.mean() if len(last3) == 3 else np.nan
        std3 = last3.std(ddof=0) if len(last3) == 3 else np.nan
        rows.append({"id": row.id, "recent": recent, "mean3": mean3,
                     "std3": std3, "kind": row.kind})
    return pd.DataFrame(rows).set_index("id")

train_x = make_features(records, train_targets)
val_x = make_features(records, val_targets)
print("훈련 특성:\n", train_x.round(3))
print("\n검증 특성:\n", val_x.round(3))
assert np.allclose(train_x.loc["a4", ["recent", "mean3", "std3"]].astype(float),
                   [6, 4, np.sqrt(8/3)])
assert np.isnan(train_x.loc["b4", "recent"])
assert val_x.loc["c6", "recent"] == 8 and val_x.loc["c7", "recent"] == 12

### 틀린 경계를 찾아봅시다

다음은 **사용하지 않을 반례**입니다. 개체를 나누지 않은 forward-fill은 앞 개체의 값을 다른 개체에 옮길 수 있습니다. 또 전체 원본에서 가운데를 중심으로 윈도를 만든 다음 나누면, 훈련 시각 2의 윈도에 검증 시각 3의 값 100이 들어갑니다. 분할 전에 만들었다고 해서 모든 rolling이 항상 잘못이라는 뜻은 아닙니다. 그러나 분할 뒤에도 사용할 수 있는 같은 개체의 과거만 포함했는지 검증하기 어렵고, 아래처럼 미래를 넣은 뒤 분할해도 누수는 되돌려지지 않습니다.

In [ ]:
wrong = pd.DataFrame({"unit": ["X", "Y", "Y"], "time": [1, 1, 2],
                      "reading": [99.0, np.nan, 2.0]})
wrong["global_fill"] = wrong.reading.ffill()  # 틀림: 다른 개체에서 복사
wrong["within_unit"] = wrong.groupby("unit").reading.ffill()
print(wrong)
assert wrong.loc[1, "global_fill"] == 99 and pd.isna(wrong.loc[1, "within_unit"])

timeline = pd.Series([2.0, 4.0, 100.0], index=[1, 2, 3])
centered_before_split = timeline.rolling(3, center=True).mean()
print("틀린 시각 2 중심 윈도 평균:", centered_before_split.loc[2])
assert np.isclose(centered_before_split.loc[2], (2+4+100)/3)
# 시각 2 예측에서 시각 2의 측정과 미래 시각 3의 측정은 모두 금지입니다.

## 3. 훈련으로만 중앙값·범위·범주를 배웁니다

결측 대체·스케일과 범주 목록처럼 자료에서 **배우는 규칙**은 훈련 예측 행으로만 `fit`합니다. 검증은 이미 배운 규칙으로 `transform`합니다. 과거 문맥을 이용해 각 행에 값을 계산하는 것과, 검증 예측 행을 전처리 규칙 학습에 섞는 것은 다릅니다.

훈련 `recent`는 6, 4, 결측, 2입니다. 관측값을 정렬하면 2, 4, 6이므로 중앙값은 4입니다. 결측을 4로 바꾼 훈련의 최솟값 $a=2$, 최댓값 $b=6$입니다. $a<b$이면 최소·최대 변환은

$$x' = \frac{x-a}{b-a}.$$

예를 들어 훈련 6은 1, 검증 8은 $(8-2)/(6-2)=1.5$입니다. 기본 설정 `clip=False`에서 검증 값은 0~1 밖이 될 수 있습니다. 학습값이 모두 같아 $a=b$이면 이 식을 그대로 나눌 수 없으므로 도구의 상수 열 처리와 문제 조건을 따로 확인합니다. 이 방법은 앞 교안의 평균·표준편차를 쓰는 **표준화**와 다르며, 각 행 벡터의 길이를 1로 만드는 행 단위 정규화와도 다릅니다. 중앙값은 관측값의 순서로 정하며 관측 개수가 짝수이면 가운데 두 값의 평균입니다. 훈련 열 전체가 결측이면 중앙값을 정할 수 없으므로 별도 정책이 필요합니다.

종류 `blue, red`는 훈련에서 정한 두 열입니다. `green`은 훈련에 없으므로 기본 설정은 오류를 냅니다. `handle_unknown="ignore"`를 명시하면 이 특성의 해당 one-hot 열이 모두 0이 됩니다. `drop=None`인 이 예제에서는 알려진 두 범주 중 하나를 나타내지 않습니다. 만약 `drop="first"`처럼 범주 하나를 생략한다면 영벡터가 생략된 알려진 범주와 미지 범주 모두를 뜻할 수 있어 구별이 모호해집니다.

In [ ]:
numeric = Pipeline([
    ("median", SimpleImputer(strategy="median")),
    ("range", MinMaxScaler()),  # 기본 clip=False
])
transformer = ColumnTransformer([
    ("num", numeric, ["recent", "mean3", "std3"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False), ["kind"]),
])
train_array = transformer.fit_transform(train_x)  # 훈련에서만 학습
val_array = transformer.transform(val_x)          # 학습된 규칙 그대로
names = transformer.get_feature_names_out()
train_encoded = pd.DataFrame(train_array, index=train_x.index, columns=names)
val_encoded = pd.DataFrame(val_array, index=val_x.index, columns=names)
print("훈련 통계 (중앙값):", transformer.named_transformers_["num"].named_steps["median"].statistics_)
print("훈련 최솟값:", transformer.named_transformers_["num"].named_steps["range"].data_min_)
print("훈련 범주:", transformer.named_transformers_["cat"].categories_)
print("검증 변환:\n", val_encoded.round(3))
assert np.isclose(transformer.named_transformers_["num"].named_steps["median"].statistics_[0], 4)
assert np.isclose(val_encoded.loc["c6", "num__recent"], 1.5)
assert np.isclose(val_encoded.loc["d6", "num__recent"], -0.25)
assert val_encoded.loc["c6", ["cat__kind_blue", "cat__kind_red"]].tolist() == [0, 0]

In [ ]:
# 검증의 과거 입력을 바꾸어도 이미 학습한 훈련 통계는 바뀌지 않습니다.
imputer = transformer.named_transformers_["num"].named_steps["median"]
scaler = transformer.named_transformers_["num"].named_steps["range"]
before = (imputer.statistics_.copy(), scaler.data_min_.copy(), scaler.data_max_.copy())
changed = records.copy()
changed.loc[(changed.unit == "C") & (changed.time == 6), "reading"] = 100.0
changed_val = make_features(changed, val_targets)
changed_encoded = transformer.transform(changed_val)  # 다시 fit하지 않음
assert np.isclose(changed_val.loc["c6", "recent"], 8)  # 현재 시각은 제외
assert np.isclose(changed_val.loc["c7", "recent"], 100)
assert all(np.array_equal(a, b) for a, b in zip(before,
    (imputer.statistics_, scaler.data_min_, scaler.data_max_)))
print("바뀐 검증 recent:", changed_val.recent.tolist())
print("검증 변경 후에도 훈련 중앙값:", imputer.statistics_[0])

try:
    OneHotEncoder().fit(train_x[["kind"]]).transform(val_x[["kind"]])
except ValueError as exc:
    print("기본 미지 범주 정책:", type(exc).__name__)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # 알려진 drop/미지 범주 경고를 출력에서만 생략
    dropped = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    dropped.fit(train_x[["kind"]])
    print("생략된 blue와 미지 green:", dropped.transform(pd.DataFrame({"kind": ["blue", "green"]})).tolist())

## 4. 점수·임계값과 ROC

이제 검증 행을 평가합니다. 전처리한 `recent`에 `red` 표시가 1이면 0.25를 더한 값을 **정해 둔 장난감 점수**로 사용합니다. 이 규칙은 모델이 학습한 것도, 실제 확률이나 성능에 대한 주장도 아닙니다. 계산된 윈도 열은 이 간단한 점수에는 사용하지 않습니다. 점수가 클수록 `+1` 쪽이라고 약속하고, $score\geq threshold$이면 `+1`로 판정합니다.

검증의 행 순서 c6,c7,d6,d7에 대한 점수는 1.5, 2.5, 0, 1.5이고 정답은 +1, -1, -1, +1입니다. 임계값 1.5에서는 동점 두 행이 **함께** 양성이 되어 TP=2, FP=1, TN=1, FN=0입니다. 행 수는 4이므로 정확도 $3/4$, 정밀도 $2/3$, 재현율 $2/2=1$입니다.

$$F1=\frac{2TP}{2TP+FP+FN}=\frac{4}{5},\quad
TPR=\frac{TP}{TP+FN},\quad FPR=\frac{FP}{FP+TN}.$$

F1은 양성의 정밀도와 재현율을 함께 보는 지표이고 TPR은 재현율입니다. 정밀도는 양성 예측이 없으면, 재현율과 TPR은 실제 양성이 없으면, FPR은 실제 음성이 없으면 각각 분모가 0이므로 정의되지 않습니다. 위 F1 식도 분모가 0이면 정의되지 않습니다. 도구가 경고와 함께 0 또는 NaN 같은 보고값을 주더라도 수학의 정의 불가와 구분합니다.

In [ ]:
scores = (val_encoded["num__recent"] + 0.25 * val_encoded["cat__kind_red"]).to_numpy()
truth = val_targets.label.to_numpy()
assert np.allclose(scores, [1.5, 2.5, 0, 1.5])

def counts_at(y, score, threshold):
    predicted = np.where(score >= threshold, 1, -1)
    return {
        "TP": int(((y == 1) & (predicted == 1)).sum()),
        "FP": int(((y == -1) & (predicted == 1)).sum()),
        "TN": int(((y == -1) & (predicted == -1)).sum()),
        "FN": int(((y == 1) & (predicted == -1)).sum()),
    }

def ratio(numerator, denominator):
    return numerator / denominator if denominator else None  # None은 정의 불가

def rates(c):
    tp, fp, tn, fn = (c[k] for k in ("TP", "FP", "TN", "FN"))
    return {"accuracy": ratio(tp + tn, tp + fp + tn + fn),
            "precision": ratio(tp, tp + fp), "recall": ratio(tp, tp + fn),
            "F1": ratio(2*tp, 2*tp + fp + fn),
            "TPR": ratio(tp, tp + fn), "FPR": ratio(fp, fp + tn)}

for threshold in [np.inf, 2.5, 1.5, 0]:
    c = counts_at(truth, scores, threshold)
    print("임계값", threshold, c, rates(c))
fpr, tpr, thresholds = roc_curve(truth, scores, pos_label=1, drop_intermediate=False)
print("ROC: (임계값, FPR, TPR)", list(zip(thresholds, fpr, tpr)))
assert np.allclose(fpr, [0, .5, .5, 1]) and np.allclose(tpr, [0, 0, 1, 1])
assert rates(counts_at(truth, scores, 1.5))["F1"] == 4/5

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.plot(fpr, tpr, marker="o", color="navy")
for x, y, threshold in zip(fpr, tpr, thresholds):
    label = "inf" if np.isinf(threshold) else f"{threshold:g}"
    ax.annotate(label, (x, y), xytext=(5, 5), textcoords="offset points")
ax.set(xlabel="FPR", ylabel="TPR", title="Validation ROC",
       xlim=(-0.05, 1.05), ylim=(-0.05, 1.05))
ax.grid(True, alpha=0.3)
plt.show()

ROC는 임계값을 높은 값에서 낮추면서 구한 **가로축 FPR, 세로축 TPR**의 점을 잇습니다. 맨 처음 $+\infty$는 모두 음성인 점입니다. 동점 1.5의 두 행은 같은 임계값에서 함께 판정되므로 그 사이에 한 행만 양성으로 바꾼 점은 이 점수 규칙의 ROC 점이 아닙니다. 그래프의 선은 계산된 점들을 연결한 것으로, 선 위의 모든 중간점이 실제 임계값에서 나온다는 뜻은 아닙니다. `roc_curve`는 양성 점수에 확률뿐 아니라 임계값 적용 전의 결정 점수도 받습니다. 실제 양성 또는 음성이 없는 자료에서는 각각 TPR 또는 FPR을 만들 수 없어 온전한 ROC를 해석할 수 없습니다.

임계값과 후보는 훈련과 **검증** 결과로 선택합니다. 같은 검증을 계속 보고 수많은 규칙을 고르면, 검증에만 우연히 잘 맞도록 선택되는 **검증 과대적합**이 생길 수 있습니다. `검증을 보고 선택한다`는 사실만으로 즉시 금지된 누수는 아닙니다. 반면 최종 테스트나 예측 시점에 없는 미래 정보를 학습·선택에 넣으면 그 평가에 대한 누수입니다. 둘 다 새 자료의 성능을 과신하게 할 수 있지만 정보가 들어온 경로가 다릅니다. 모든 후보와 임계값을 고정한 뒤 보류한 최종 테스트로 평가합니다. 테스트 결과를 보고 다시 설정을 바꾸었다면 그 테스트는 더 이상 독립 최종 평가가 아닙니다.

## 5. 무엇을 맞히는 데 얼마의 가중치를 둘까요?

평가 기준이 정답에 따라 가중치를 달리 정할 수도 있습니다. 아래의 새 합성 표에서 실제 배경 B의 가중치는 1, 실제 대상 T의 가중치는 5입니다. 가중치는 **예측이 아니라 실제 정답**으로 정하며, 정답을 알기 전 예측 특성으로 쓰는 것이 아닙니다.

$$\text{가중 정확도}=
\frac{1\times(맞힌\ B\ 수)+5\times(맞힌\ T\ 수)}
{1\times(전체\ 실제\ B\ 수)+5\times(전체\ 실제\ T\ 수)}.$$

| 전체 실제 정답 | 후보 | 맞힌 B | 맞힌 T | 전체 행 정확도 | 가중 정확도 |
|---|---|---:|---:|---:|---:|
| B 5개, T 1개 | 가 | 5 | 0 | $5/6$ | $5/10$ |
| B 5개, T 1개 | 나 | 3 | 1 | $4/6$ | $8/10$ |

분모는 맞힌 행만이 아니라 **모든 평가 행의 최대 가중점수**입니다. 같은 두 후보의 순위가 기준에 따라 바뀝니다. 이것만으로 어떤 기준이 언제나 낫다고 정할 수 없습니다. 실제 문제에서는 공식 점수와 평가 단위를 확인합니다.

[IOAI 2025 Radar 공식 문제](https://raw.githubusercontent.com/IOAI-official/IOAI-2025/main/Individual-Contest/Radar/Radar.ipynb)는 실제 배경과 비배경을 다르게 가중해 전체 가능한 점수로 나눕니다. 위 숫자와 표는 설명을 위한 별도 합성 예이며 이 국제 문제가 국내 중등 기출이라는 뜻이 아닙니다.

## 6. 새 입력으로 직접 확인하기

새 합성 자료의 한 예측 행 역시 측정 **전**에 만들어집니다. 종류는 미리 알려지고 정답은 그날 뒤에 알려집니다. 이전 관측은 같은 개체의 해당 시각보다 엄격히 앞선 것만 허용합니다. 아래 표는 모두 원본 측정이고, 빈 칸은 결측입니다. 각 행의 측정은 그 시각의 예측 후에 도착합니다.

| 개체 | 시각 1 | 시각 2 | 시각 3 | 시각 4 | 시각 6 | 종류 |
|---|---:|---:|---:|---:|---:|---|
| M | 5 | 7 | 9 | 결측 | — | gold |
| N | 결측 | 결측 | 결측 | 1 | — | silver |
| P | 7 | 9 | 11 | 결측 | 17 | bronze |
| Q | 결측 | 0 | 결측 | 결측 | 11 | gold |

예측 행 M4, M5, N4, N5는 훈련이고, P6, P7, Q6, Q7은 검증입니다. 훈련 정답은 시각 6 전에 모두 알려졌다고 가정합니다. P/Q의 허용된 과거는 예측 서비스에 이미 도착한 라벨 없는 같은 개체의 문맥입니다. 검증 정답은 순서대로 $[+1,-1,+1,-1]$입니다. 최종 테스트의 새 개체와 정답은 별도 보류합니다.

1. 각 행의 `recent`를 구하고, M4와 P6의 최근 관측 세 개·평균·모집단 표준편차를 계산하세요. N4와 Q6에서 forward-fill의 출처를 설명하세요. P7에 시각 6 측정을 쓸 수 있는 조건과 한꺼번에 예측하는 계약에서의 차이도 설명하세요.
2. 훈련 `recent`의 중앙값과 결측 대체 후 최솟값·최댓값을 구하고 검증 네 값을 변환하세요. 훈련 종류로 one-hot 열을 학습했을 때 `bronze`와 `gold`는 각각 무엇이 되나요? `drop="first"`가 미지 범주 해석에 주는 영향도 설명하세요.
3. 점수는 변환한 `recent` 자체이고 클수록 양성입니다. 이 자료는 훈련 범위가 8이어서 아래 값과 임계값을 이진 부동소수로 정확히 표현할 수 있습니다. 다른 자료의 경계는 부동소수 오차를 확인하고 비교 정책을 정해야 합니다. 점수가 임계값 이상이면 양성입니다. 임계값 $2$, $5/4$, $-1/8$에서 TP/FP/TN/FN, 정확도·정밀도·재현율·F1, FPR/TPR을 구하세요. ROC 좌표를 임계값이 높은 순으로 적어 가로축 FPR, 세로축 TPR에 그린 뒤 동점이 한 번에 바뀌는지 확인하세요. 실제 양성 또는 음성이 없는 표와 양성 예측이 없는 표에서 어떤 분모가 0인지 쓰세요.
4. 아래 코드는 왜 틀렸나요? 새 자료로 고치고, **검증 입력을 바꾸어도 훈련 중앙값·범위·범주가 불변**임을 코드로 확인하세요. 동시에 그 입력을 과거로 쓸 수 있는 검증 행의 변환값은 바뀔 수 있음을 확인하세요.
```python
# 잘못된 예: 평가할 자료를 규칙 학습에 섞음
encoder.fit(pd.concat([train_x, val_x])[["kind"]])
```
5. (가) 검증 기록의 미래 값을 중심 윈도에 넣은 경우, (나) 훈련만 사용해 만든 많은 후보 중 동일 검증 성적이 가장 좋은 것을 계속 고른 경우, (다) 최종 테스트 정답으로 임계값을 고른 경우의 정보 흐름을 비교하세요. 누수와 검증 반복 사용의 과대적합 가능성을 구분하고 각 평가에 미치는 영향을 설명하세요.
6. 별도 합성 평가에서 실제 B 6개·T 2개, 가중치는 실제 B에 1, 실제 T에 4입니다. 후보 U는 B 6개와 T 0개, 후보 V는 B 3개와 T 2개를 맞혔습니다. 두 후보의 전체 행 정확도와 가중 정확도를 계산하고 순위가 같은지 비교하세요. 가중 분모에 어떤 행이 포함되는지 설명하세요.

직접 새 DataFrame을 만들고 분할, 특성 생성, `fit`/`transform`, 임계값별 계산을 작성하세요. 결과를 적을 때 행 ID·점수 순서와 사용한 도움을 함께 남기세요.

In [ ]:
# 새 자료를 직접 입력하고 코드를 작성하세요.

## 참고 자료

- [KOAI/IOAI 출제 기준](https://kitpa.org/koai/syllabus): 해당 국내 페이지의 데이터 전처리, 특성, 지표와 ROC 항목. 페이지에는 IOAI 2026 기준이라고 표시되어 있습니다.
- [scikit-learn: 전처리와 누수](https://scikit-learn.org/stable/common_pitfalls.html), [최소·최대 변환](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html), [one-hot](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [ROC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html).
- [pandas: 그룹 처리](https://pandas.pydata.org/docs/user_guide/groupby.html), [forward-fill](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ffill.html).